# Data Quality Validation

This notebook reads `data_quality_rules` metadata to dynamically validate data in the silver layer. Each rule defines a check type (not_null, unique, range, regex, custom_sql), the column to check, and the severity level. Results are logged to `pipeline_run_log` and a DQ results table.

In [0]:
# Create DQ results table (no DEFAULT clause for Delta compat)
spark.sql("""
CREATE TABLE IF NOT EXISTS metadata_framework.bronze.dq_results (
  dq_result_id    STRING    NOT NULL COMMENT 'Unique result ID (UUID)',
  dq_rule_id      STRING    NOT NULL COMMENT 'FK to data_quality_rules',
  table_id        STRING    NOT NULL COMMENT 'FK to table_config',
  rule_name       STRING    NOT NULL COMMENT 'Name of the DQ rule',
  rule_type       STRING    NOT NULL COMMENT 'Type of DQ rule',
  column_name     STRING    COMMENT 'Column checked',
  severity        STRING    NOT NULL COMMENT 'error, warn, info',
  check_status    STRING    NOT NULL COMMENT 'pass, fail',
  total_records   BIGINT    COMMENT 'Total records checked',
  failed_records  BIGINT    COMMENT 'Records that failed the check',
  failure_pct     DOUBLE    COMMENT 'Percentage of records that failed',
  checked_at      TIMESTAMP NOT NULL COMMENT 'When the check ran'
) COMMENT 'Stores results of data quality rule evaluations.'
""")
print("dq_results table ready.")

In [0]:
import uuid
from datetime import datetime
from pyspark.sql.functions import expr

CATALOG = "metadata_framework"
META_SCHEMA = f"{CATALOG}.bronze"
SILVER_SCHEMA = f"{CATALOG}.silver"

def get_dq_rules(table_id):
    """Fetch all active DQ rules for a given table_id."""
    return (spark.read.table(f"{META_SCHEMA}.data_quality_rules")
            .where(f"table_id = '{table_id}' AND is_active = true")
            .collect())

def get_table_config(table_id):
    """Fetch table configuration."""
    return (spark.read.table(f"{META_SCHEMA}.table_config")
            .where(f"table_id = '{table_id}'")
            .collect()[0])

def log_pipeline_run(run_id, table_id, status, layer, records_read=0, records_written=0, records_rejected=0, error_message=None):
    """Insert a record into pipeline_run_log."""
    log_df = spark.createDataFrame([(
        run_id, table_id, status, layer,
        records_read, records_written, records_rejected,
        error_message,
        datetime.now(),
        datetime.now() if status in ("success", "failed") else None
    )], schema="run_id STRING, table_id STRING, run_status STRING, run_layer STRING, records_read BIGINT, records_written BIGINT, records_rejected BIGINT, error_message STRING, started_at TIMESTAMP, completed_at TIMESTAMP")
    log_df.write.mode("append").saveAsTable(f"{META_SCHEMA}.pipeline_run_log")

def log_dq_result(dq_rule_id, table_id, rule_name, rule_type, column_name, severity, status, total_records, failed_records):
    """Log a DQ check result into dq_results table."""
    failure_pct = (failed_records / total_records * 100.0) if total_records > 0 else 0.0
    result_df = spark.createDataFrame([(
        str(uuid.uuid4()), dq_rule_id, table_id, rule_name, rule_type,
        column_name, severity, status, total_records, failed_records, failure_pct,
        datetime.now()
    )], schema="dq_result_id STRING, dq_rule_id STRING, table_id STRING, rule_name STRING, rule_type STRING, column_name STRING, severity STRING, check_status STRING, total_records BIGINT, failed_records BIGINT, failure_pct DOUBLE, checked_at TIMESTAMP")
    result_df.write.mode("append").saveAsTable(f"{META_SCHEMA}.dq_results")

print("DQ utilities loaded.")

In [0]:
def run_dq_check(rule, table_name, table_id):
    """
    Execute a single DQ rule against a silver table and return the result.
    """
    rule_type = rule.rule_type
    column = rule.column_name
    rule_expr = rule.rule_expression
    rule_name = rule.rule_name
    severity = rule.severity
    dq_rule_id = rule.dq_rule_id
    
    total_records = spark.read.table(table_name).count()
    
    failed_records = 0
    
    if rule_type == "not_null":
        failed_records = spark.sql(f"SELECT COUNT(*) FROM {table_name} WHERE {column} IS NULL").collect()[0][0]
    
    elif rule_type == "unique":
        total_count = spark.sql(f"SELECT COUNT(*) FROM {table_name}").collect()[0][0]
        distinct_count = spark.sql(f"SELECT COUNT(DISTINCT {column}) FROM {table_name}").collect()[0][0]
        failed_records = total_count - distinct_count
    
    elif rule_type == "range":
        failed_records = spark.sql(f"SELECT COUNT(*) FROM {table_name} WHERE NOT ({column} {rule_expr})").collect()[0][0]
    
    elif rule_type == "regex":
        failed_records = spark.sql(f"SELECT COUNT(*) FROM {table_name} WHERE NOT {column} RLIKE '{rule_expr}'").collect()[0][0]
    
    elif rule_type == "custom_sql":
        # rule_expression contains a full SQL that should return a count of failures
        failed_records = spark.sql(f"SELECT COUNT(*) FROM {table_name} WHERE NOT ({rule_expr})").collect()[0][0]
    
    status = "pass" if failed_records == 0 else "fail"
    
    icon = "✅" if status == "pass" else ("⚠️" if severity == "warn" else "❌")
    print(f"  {icon} {rule_name}: {status} ({failed_records}/{total_records} failed) [{severity}]")
    
    log_dq_result(dq_rule_id, table_id, rule_name, rule_type, column, severity, status, total_records, failed_records)
    
    return status, severity, failed_records

def validate_table(table_id):
    """Run all DQ rules for a given table."""
    config = get_table_config(table_id)
    silver_table = f"{SILVER_SCHEMA}.{config.target_table_name.replace('raw_', '')}"
    
    # Check if silver table exists
    if not spark.catalog.tableExists(silver_table):
        print(f"Silver table {silver_table} does not exist yet, skipping DQ checks.")
        return []
    
    rules = get_dq_rules(table_id)
    if not rules:
        print(f"No active DQ rules for {table_id}, skipping.")
        return []
    
    run_id = str(uuid.uuid4())
    print(f"\n{'='*60}")
    print(f"Validating: {silver_table} ({len(rules)} rules)")
    log_pipeline_run(run_id, table_id, "started", "silver")
    
    results = []
    total_failures = 0
    has_errors = False
    
    for rule in rules:
        status, severity, failed = run_dq_check(rule, silver_table, table_id)
        results.append((status, severity))
        total_failures += failed
        if status == "fail" and severity == "error":
            has_errors = True
    
    run_status = "success" if not has_errors else "failed"
    log_pipeline_run(run_id, table_id, run_status, "silver", 0, 0, total_failures)
    
    return results

print("DQ check functions defined.")

In [0]:
# ============================================================================
# Execute DQ validation for all active tables
# ============================================================================
active_tables = (spark.read.table(f"{META_SCHEMA}.table_config")
                 .where("is_active = true")
                 .select("table_id")
                 .collect())

all_results = []
for row in active_tables:
    results = validate_table(row.table_id)
    all_results.extend(results)

# Summary
passed = sum(1 for s, _ in all_results if s == "pass")
failed = sum(1 for s, _ in all_results if s == "fail")
print(f"\n{'='*60}")
print(f"DQ Summary: {passed} passed, {failed} failed out of {len(all_results)} total checks")
if failed > 0:
    print("⚠️  Some data quality checks failed — review the dq_results table for details.")